In [ ]:
from pathlib import Path
import os

# Set PROJECT_DATA_DIR before launching Jupyter to use data stored elsewhere.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".gitignore").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = Path(os.environ.get("PROJECT_DATA_DIR", str(PROJECT_ROOT / "data"))).expanduser()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    balanced_accuracy_score, roc_auc_score,
    classification_report, confusion_matrix
)
import seaborn as sns
import matplotlib.pyplot as plt
import shap
from catboost import CatBoostClassifier, Pool


df = pd.read_csv(str(DATA_DIR / 'AI_UNITE_CRSS_FARS_merged_subset_DRIMPAIR_ONLY.csv'), encoding = 'latin-1')

df = df.dropna(subset=['INJ_SEV'])
df['INJ_SEV'] = df['INJ_SEV'].astype(int)
target = 'INJ_SEV'


categorical_cols = [
    'PERNOTMVIT','PVH_INVL','PERMVIT','MONTH','DAY_WEEK','YEAR',
    'HARM_EV','MAN_COLL','TYP_INT','REL_ROAD','WRK_ZONE','LGT_COND','WEATHER',
    'DRDISTRACT','DRIMPAIR','SPEC_USE','SEX','REST_USE','REST_MIS',
    'HELM_USE','HELM_MIS','DRINKING','ALC_STATUS','ATST_TYP',
    'ALC_RES','DRUGS','STR_VEH','LOCATION','VE_FORMS','HIT_RUN','BODY_TYP','TOW_VEH',
    'CARGO_BT','HAZ_INV','EMER_USE','DR_PRES','SPEEDREL','VTRAFWAY',
    'VSURCOND','VISION', 'VSPD_LIM', 'VE_TOTAL', 'PEDS' # , 'AIR_BAG','EJECTION',
] 
numeric_cols = ['TRAV_SP', 'AGE']

all_columns = categorical_cols + numeric_cols + [target]

for col in all_columns:
    df[col] = df[col].astype('string')

X = df[categorical_cols + numeric_cols]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

train_pool = Pool(X_train, y_train, cat_features=categorical_cols)
test_pool  = Pool(X_test,  y_test,  cat_features=categorical_cols)


model = CatBoostClassifier(
    loss_function='MultiClass',          
    eval_metric='Accuracy',              
    iterations=1000,                     
    learning_rate=0.05,
    depth=8,                             
    l2_leaf_reg=3,                       
    random_strength=1.0,                 
    auto_class_weights='Balanced',       
    bootstrap_type='Bayesian',           
    task_type='GPU',                     
    verbose=200
)

model.fit(
    train_pool,
    eval_set=test_pool,
    use_best_model=True,
    early_stopping_rounds=50
)

model.fit(
    train_pool,
    eval_set=test_pool,
    use_best_model = True,
    early_stopping_rounds=100,
)

y_pred = model.predict(X_test)

confusion = confusion_matrix(y_test, y_pred)
print(classification_report(y_test, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))



plt.imshow(confusion, cmap='binary', interpolation='nearest')
plt.colorbar()
tick_marks = np.arange(2)
thresh = confusion.max() / 2.
for i in range(confusion.shape[0]):
    for j in range(confusion.shape[1]):
        plt.text(j, i, format(confusion[i, j]), ha="center", va="center", color="white" if confusion[i, j] > thresh else "black")


In [ ]:
print(X_test.dtypes.head(30))
print([c for c in X_test.columns if X_test[c].dtype == 'object'])
print([c for c in X_test.columns if str(X_test[c].dtype).startswith('category')])


In [ ]:
import shap
import numpy as np

explainer = shap.TreeExplainer(model)

shap_sample = X_test.iloc[:100] 

shap_values = explainer.shap_values(shap_sample)

shap.summary_plot(shap_values, shap_sample, plot_type="bar", max_display=20)
